In [19]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csgraph
from sklearn.manifold import SpectralEmbedding
from sklearn.preprocessing import normalize

import numpy as np
import torch
from PIL import Image
import os
import pandas as pd

#importing the used models
from transformers import BlipProcessor, BlipModel

In [42]:
def load_and_filter_data(caption_csv_path):
    img_dir="./images"
    img_prefix="img_"
    df = pd.read_csv(caption_csv_path, encoding='latin1')
    df['image_filename'] = [f"{img_prefix}{i}.jpg" for i in df.index]
    df['full_image_path'] = [f"{img_dir}/{fname}" for fname in df['image_filename']]
    df_valid = df[df['full_image_path'].apply(os.path.exists)].reset_index(drop=True)
    print(f"Valid images with captions: {len(df_valid)}")
    return df_valid


def generate_image_embeddings(image_paths, processor, model, device):
    embeddings = []
    for path in image_paths:
        try:
            image = Image.open(path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_image_features(**inputs)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            embeddings.append(features.cpu().numpy().squeeze())
        except Exception as e:
            print(f"Error processing image {path}: {e}")
    return np.array(embeddings)

def generate_text_embeddings(texts, processor, model, device):
    embeddings = []
    for text in texts:
        try:
            inputs = processor(text=text, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_text_features(**inputs)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            embeddings.append(features.cpu().numpy().squeeze())
        except Exception as e:
            print(f"Error processing text '{text[:30]}...': {e}")
    return np.array(embeddings)

def match_embeddings(image_embeddings, text_embeddings, valid_image_paths, valid_captions):
    similarity_matrix = cosine_similarity(image_embeddings, text_embeddings)
    print(similarity_matrix)
    matched_pairs = []
    for i in range(similarity_matrix.shape[0]):
        j = np.argmax(similarity_matrix[i])
        matched_pairs.append({
            "image_path": valid_image_paths[i],
            "caption": valid_captions[j],
            "image_embedding": image_embeddings[i],
            "text_embedding": text_embeddings[j],
            "similarity": similarity_matrix[i, j]
        })
    print(f"Matched {len(matched_pairs)} image-caption pairs via cosine similarity.")
    return matched_pairs, similarity_matrix

In [75]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

df_long = load_and_filter_data("./long_generated_captions.csv")
df_short = load_and_filter_data("./short_generated_captions.csv")

valid_image_paths = df_long['full_image_path'].tolist()
valid_captions_long = df_long['paraphrased_caption'].tolist()
valid_captions_short = df_short['paraphrased_caption'].tolist()

image_embeddings = generate_image_embeddings(valid_image_paths, processor, model, device)
text_embeddings_long = generate_text_embeddings(valid_captions_long, processor, model, device)
text_embeddings_short = generate_text_embeddings(valid_captions_short, processor, model, device)
matched_pairs_long, similarity_matrix_spectral_long = match_embeddings(image_embeddings, text_embeddings_long, valid_image_paths, valid_captions_long)
matched_pairs_short, similarity_matrix_spectral_short = match_embeddings(image_embeddings, text_embeddings_short, valid_image_paths, valid_captions_short)


`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_mo

Valid images with captions: 498
Valid images with captions: 498


In [23]:
def apply_spectral_embedding_from_cosine(B, n_components=128):
    B = np.clip(B, 0, None)

    n_texts, n_images = B.shape

    zero_text = np.zeros((n_texts, n_texts))
    zero_image = np.zeros((n_images, n_images))
    
    A = np.block([
        [zero_text, B],
        [B.T, zero_image]
    ])

    epsilon = 1e-3
    A += epsilon
    np.fill_diagonal(A, epsilon)

    n_total = A.shape[0]
    if n_components >= n_total:
        n_components = n_total - 1
        print(f"Warning: n_components reduced to {n_components} because it must be less than total nodes {n_total}")

    spectral = SpectralEmbedding(n_components=n_components, affinity='precomputed')
    embedded = spectral.fit_transform(A)

    spectral_text = normalize(embedded[:n_texts])
    spectral_image = normalize(embedded[n_texts:])
    
    return spectral_image, spectral_text


In [24]:
spectral_image_embeddings, spectral_text_long_embeddings = apply_spectral_embedding_from_cosine(similarity_matrix_spectral_long)
spectral_image_embeddings, spectral_text_short_embeddings = apply_spectral_embedding_from_cosine(similarity_matrix_spectral_short)

## Generate and add vector embeddings to Qdrant
**Normal embeddings of original_captions column**

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

df_valid = load_and_filter_data()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.

Valid images with captions: 423


In [5]:
df_valid.head()

,Unnamed: 0,uid,url,key,status,original_caption,vlm_model,vlm_caption,toxicity,severe_toxicity,...,original_width,original_height,exif,sha256,image_id,author,subreddit,score,image_filename,full_image_path
0,2,f3a6ad643814199a98f872229758b8e735773c1803e2f1...,https://images.pexels.com/photos/770224/pexels...,7740009,success,Water Coming Out Of A Pipe,gemini-pro-vision,This image displays: A large rusty pipe with ...,0.002487,0.000007,...,500,750,{},f3a6ad643814199a98f872229758b8e735773c1803e2f1...,NaN,NaN,-1,-1,img_1.jpg,./images/img_1.jpg
1,3,3b03f5e3a32499281dbeebb9a0e1d3dfe883bc7b72e2eb...,https://filmfare.wwmindia.com/content/2017/jul...,7740010,success,MARCH 2009 ''It's sad that a boy and girl can ...,gemini-pro-vision,This image displays: three pictures of Katrin...,0.001743,0.000003,...,600,450,{},3b03f5e3a32499281dbeebb9a0e1d3dfe883bc7b72e2eb...,NaN,NaN,-1,-1,img_2.jpg,./images/img_2.jpg
2,4,9f3b5395744728e68b85083c0411c3b4139214f1258bf7...,https://southernersays.com/wp-content/uploads/...,7740000,success,Humpback Whale tail in the Bay of Banderas,gemini-pro-vision,This image displays:\nA photograph of a whale...,0.006152,0.000024,...,1024,662,{},9f3b5395744728e68b85083c0411c3b4139214f1258bf7...,NaN,NaN,-1,-1,img_3.jpg,./images/img_3.jpg
3,5,e3c7435ce411fea5656c792355a3811163da286673f8e5...,https://i.pinimg.com/originals/9a/f5/a6/9af5a6...,7740035,success,"The gaming Team logo of team DUO Team Logo, My...",gemini-pro-vision,"This image displays:\n\nText that reads ""20% ...",0.007788,0.000023,...,1920,1080,{},e3c7435ce411fea5656c792355a3811163da286673f8e5...,NaN,NaN,-1,-1,img_4.jpg,./images/img_4.jpg
4,6,3d652d76feea1b487781bdb6a3d2cf961a4e52097b3b70...,https://themaydan.com/wp-content/uploads/2019/...,7740030,success,"Tea, chilies, and takeaway: what food choices ...",gemini-pro-vision,This image displays: A family of four is sitt...,0.000606,0.000002,...,926,618,{},3d652d76feea1b487781bdb6a3d2cf961a4e52097b3b70...,NaN,NaN,-1,-1,img_5.jpg,./images/img_5.jpg


In [7]:
df_valid.columns

Index(['Unnamed: 0', 'uid', 'url', 'key', 'status', 'original_caption',
       'vlm_model', 'vlm_caption', 'toxicity', 'severe_toxicity', 'obscene',
       'identity_attack', 'insult', 'threat', 'sexual_explicit',
       'watermark_class_id', 'watermark_class_score', 'aesthetic_score',
       'error_message', 'width', 'height', 'original_width', 'original_height',
       'exif', 'sha256', 'image_id', 'author', 'subreddit', 'score',
       'image_filename', 'full_image_path'],
      dtype='object')

In [15]:
df_valid = df_valid.rename(columns={'Unnamed: 0': "img_idx"})

In [16]:
df_valid.columns

Index(['img_idx', 'uid', 'url', 'key', 'status', 'original_caption',
       'vlm_model', 'vlm_caption', 'toxicity', 'severe_toxicity', 'obscene',
       'identity_attack', 'insult', 'threat', 'sexual_explicit',
       'watermark_class_id', 'watermark_class_score', 'aesthetic_score',
       'error_message', 'width', 'height', 'original_width', 'original_height',
       'exif', 'sha256', 'image_id', 'author', 'subreddit', 'score',
       'image_filename', 'full_image_path'],
      dtype='object')

### Add to Qdrant

In [6]:
df_s_captions = pd.read_csv("./short_generated_captions.csv")

In [7]:
valid_captions = df_s_captions['original_caption'].tolist()
img_ids = df_s_captions["id"].tolist()
# needed as our dataset saved from 1 and not 0
valid_img_idx = [id+1 for id in img_ids]
valid_img_paths = [f'./images/img_{id}.jpg' for id in valid_img_idx]

print(valid_img_idx)
print(valid_captions)
print(valid_img_paths)

image_embeddings = generate_image_embeddings(valid_img_paths[:10], processor, model, device)
text_embeddings = generate_text_embeddings(valid_captions[:10], processor, model, device)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 22

In [8]:
collection_name = "normal_original_captions"

client = QdrantClient(host="localhost", port=6333)

points = [
    PointStruct(id=idx, vector=vec, payload={})
    for idx, vec in zip(valid_img_idx, image_embeddings)
]

client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [68]:
# query from qdrant

def convert_to_text_embedding(user_prompt):
    inputs = processor(text=user_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        features = model.get_text_features(**inputs)
    features = features / features.norm(p=2, dim=-1, keepdim=True)

    return features.cpu().numpy().squeeze()

In [69]:
caption_to_test = df_s_captions.loc[3]["paraphrased_caption"]

In [70]:
caption_to_test

"Humpback whale's tail in Banderas Bay."

In [72]:
test_cap_embed = convert_to_text_embedding(caption_to_test)
search_result = client.query_points(
    collection_name="normal_original_captions",
    query=test_cap_embed,
    with_payload=False,
    limit=5
).points

print(search_result)

[ScoredPoint(id=3, version=0, score=0.07833718, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=8, version=0, score=0.06000434, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=6, version=0, score=0.048723906, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=0, score=0.0447171, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=7, version=0, score=0.037586916, payload=None, vector=None, shard_key=None, order_value=None)]


In [73]:
caption_to_test = df_s_captions.loc[4]["paraphrased_caption"]
test_cap_embed = convert_to_text_embedding(caption_to_test)
search_result = client.query_points(
    collection_name="normal_original_captions",
    query=test_cap_embed,
    with_payload=False,
    limit=5
).points

print(search_result)

[ScoredPoint(id=8, version=0, score=0.0751812, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=0, score=0.074245825, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=6, version=0, score=0.047488242, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=0, score=0.0349559, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=7, version=0, score=0.029347159, payload=None, vector=None, shard_key=None, order_value=None)]
